# Car-GAN — DCGAN on Stanford Cars

This notebook walks through training a Deep Convolutional GAN that generates 64×64 car images.
All the logic lives in the `car_gan` package (`src/car_gan/`); the notebook just drives it and visualises the results.

**Setup** (once, from the repository root):

```bash
pip install -e ".[all]"
```

Run the notebook from the `notebooks/` folder. Paths below are resolved relative to the repository root.

In [ ]:
import os
from pathlib import Path

import torch

from car_gan import Config, Discriminator, Generator, get_default_device, init_weights, load_generator
from car_gan.data import build_dataloader, download_dataset
from car_gan.trainer import GANTrainer, plot_history
from car_gan.utils import make_video, sample_latent, show_images

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)

config = Config(epochs=100, batch_size=128, num_workers=2)
device = get_default_device()
print(f"Repository root: {ROOT}")
print(f"Device: {device}")

## 1. Download the dataset

The [Stanford Cars dataset](https://www.kaggle.com/datasets/jessicali9530/stanford-cars-dataset) (~1.8 GB) is downloaded from Kaggle.
You need a Kaggle API token: place `kaggle.json` in the repository root, or enter your username and key when prompted.

In [ ]:
download_dataset(config)

## 2. Load and inspect the data

Images are resized, centre-cropped to 64×64 and normalised to [-1, 1] to match the generator's `Tanh` output.
Both the train and test splits are used because the GAN is unconditional.

In [ ]:
dataloader = build_dataloader(config)
print(f"{len(dataloader.dataset)} images, {len(dataloader)} batches per epoch")

real_images, _ = next(iter(dataloader))
show_images(real_images)

## 3. Build the networks

See [`docs/architecture.md`](../docs/architecture.md) for a layer-by-layer breakdown.
Weights are initialised from N(0, 0.02) as recommended in the DCGAN paper.

In [ ]:
generator = Generator(config.latent_size)
discriminator = Discriminator()
generator.apply(init_weights)
discriminator.apply(init_weights)
generator

In [ ]:
discriminator

## 4. Train

Each epoch saves a sample grid (from a fixed latent batch) to `generated-images/`, and checkpoints are written to `checkpoints/` every 10 epochs.

> ⚠️ Training overwrites `checkpoints/generator.pth` and `checkpoints/discriminator.pth`.
> Point `config.checkpoint_dir` somewhere else if you want to keep the published weights.

In [ ]:
trainer = GANTrainer(generator, discriminator, config, device)
history = trainer.fit(dataloader)

## 5. Training curves

In [ ]:
plot_history(history)

## 6. Training timelapse

Stitch the per-epoch sample grids into a video.

In [ ]:
make_video(config.sample_dir, "gans_training.avi", fps=4)

## 7. Generate new cars from a checkpoint

This works without training: it loads the published 100-epoch weights from `checkpoints/`.

In [ ]:
pretrained = load_generator(config.checkpoint_dir / "generator.pth", config.latent_size, device)

with torch.no_grad():
    images = pretrained(sample_latent(64, config.latent_size, device))
show_images(images)